# v1.3 raw attempt input dataset build

This notebook defines the raw dashboard input contract for v1.3 and writes `data/raw_attempts.csv` plus validation reports.

Non-goals:
- no dashboard refactor
- no v13_pipeline.py yet
- no readiness formula changes
- no DQ filtering beyond audit
- no proxy output builds


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'dashboards').exists() and (candidate / 'data').exists():
            return candidate
    return start.resolve()


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

METRICS_PATH = REPO_ROOT / 'dashboards' / 'metrics'
if str(METRICS_PATH) not in sys.path:
    sys.path.insert(0, str(METRICS_PATH))

DATA_DIR = REPO_ROOT / 'data'
OUTPUT_PATH = DATA_DIR / 'raw_attempts.csv'
SCHEMA_REPORT_PATH = DATA_DIR / 'raw_attempts_schema_report.csv'
VALIDATION_REPORT_PATH = DATA_DIR / 'raw_attempts_validation_report.csv'
LOCAL_SOURCE_PATH = DATA_DIR / 'dq_artifacts' / 'fact_attempts_raw.csv'
BASELINE_PATH = DATA_DIR / 'verify_df_fixed.csv'
LEGACY_PATH = DATA_DIR / 'old_verify_df_fixed.csv'

SOURCE_MODE = os.getenv('RAW_ATTEMPTS_SOURCE', 'local').strip().lower()
DB_SCHEMA = os.getenv('DB_SCHEMA', 'public')
DB_USER_TABLE = os.getenv('DB_USER_TABLE', 'users')


In [ ]:
script_import_error = None
try:
    from scripts.v1_3_extraction_rebuild import build_engine, build_raw_attempts, comparison_report
except Exception as exc:
    build_engine = None
    build_raw_attempts = None
    comparison_report = None
    script_import_error = exc


def coalesce_aliases(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    alias_map = {
        'attempt_id': 'test_taker_id',
        'test_name': 'name',
        'duration': 'time_limit',
        'learner_id': 'username',
        'question_bank_count': 'total_questions',
        'max_marks_db': 'total_questions',
    }
    for alias, source in alias_map.items():
        if alias not in out.columns and source in out.columns:
            out[alias] = out[source]
    return out


def load_raw_attempts() -> tuple[pd.DataFrame, str]:
    if SOURCE_MODE == 'db':
        if build_raw_attempts is None or build_engine is None:
            raise RuntimeError(f'DB mode requested but the extractor script could not be imported: {script_import_error}')
        engine = build_engine()
        df = build_raw_attempts(engine, schema=DB_SCHEMA, user_table=DB_USER_TABLE)
        label = f'db:{DB_SCHEMA}.{DB_USER_TABLE}'
    else:
        if not LOCAL_SOURCE_PATH.exists():
            raise FileNotFoundError(f'Local source not found: {LOCAL_SOURCE_PATH}')
        df = pd.read_csv(LOCAL_SOURCE_PATH)
        label = f'file:{LOCAL_SOURCE_PATH.relative_to(REPO_ROOT)}'

    df = coalesce_aliases(df)
    df = df.loc[:, ~df.columns.duplicated()].copy()

    for col in ['created_at', 'updated_at', 'finished_at', 'membership_created_at', 'membership_updated_at']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    for col in ['marks', 'no_of_questions', 'time_taken', 'pass_mark', 'attempted_questions', 'correct_answers', 'total_questions', 'duration', 'question_bank_count', 'max_marks_db', 'class_id', 'subscriber_id', 'group_id']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    preferred_order = [
        'attempt_id', 'test_taker_id', 'user_id', 'test_id', 'test_name', 'name',
        'created_at', 'finished_at', 'updated_at',
        'marks', 'attempted_questions', 'correct_answers', 'wrong_answers',
        'time_taken', 'duration', 'no_of_questions', 'question_limit',
        'total_questions', 'question_bank_count', 'max_marks_db', 'pass_mark',
        'f_name', 'l_name', 'username', 'learner_id', 'student_id',
        'institute', 'institute_standardized', 'city', 'country',
        'class_id', 'subscriber_id', 'group_id', 'description', 'instructions', 'time_limit', 'occurrence'
    ]
    df = df[[c for c in preferred_order if c in df.columns] + [c for c in df.columns if c not in preferred_order]].copy()
    return df, label


raw_attempts, source_label = load_raw_attempts()
raw_attempts.head(3)


In [ ]:
EXPECTED_COLUMNS = [
    'attempt_id', 'test_taker_id', 'user_id', 'test_id', 'test_name', 'name',
    'created_at', 'finished_at', 'updated_at',
    'marks', 'attempted_questions', 'correct_answers', 'wrong_answers',
    'time_taken', 'duration', 'no_of_questions', 'question_limit',
    'total_questions', 'question_bank_count', 'max_marks_db', 'pass_mark',
    'f_name', 'l_name', 'username', 'learner_id', 'student_id',
    'institute', 'institute_standardized', 'city', 'country',
    'class_id', 'subscriber_id', 'group_id',
    'description', 'instructions', 'time_limit', 'occurrence',
    'delivered_result_questions', 'answer_grade_sum_diagnostic',
    'subject_id', 'topic_id', 'year_group'
]

ALIAS_OF = {
    'attempt_id': 'test_taker_id',
    'test_name': 'name',
    'duration': 'time_limit',
    'learner_id': 'username',
    'question_bank_count': 'total_questions',
    'max_marks_db': 'total_questions',
}

LOGICAL_CONTRACT = [
    ('attempt identifier', ['attempt_id', 'test_taker_id'], True, 'one attempt row is keyed by an attempt identifier'),
    ('user identifier', ['user_id'], True, 'user_id must be present'),
    ('test identifier', ['test_id'], True, 'test_id must be present'),
    ('test name', ['test_name', 'name'], True, 'raw dataset should keep the test label'),
    ('created_at', ['created_at'], True, 'attempt ordering anchor'),
    ('finished_at', ['finished_at'], True, 'completion evidence field'),
    ('marks', ['marks'], True, 'score field'),
    ('attempted_questions', ['attempted_questions'], True, 'attempt-level question evidence'),
    ('correct_answers', ['correct_answers'], True, 'attempt-level correctness evidence'),
    ('time_taken', ['time_taken'], True, 'timing evidence'),
    ('duration or limit', ['duration', 'question_limit', 'time_limit'], True, 'at least one delivered duration or limit field should exist'),
    ('pass_mark', ['pass_mark'], True, 'threshold evidence'),
    ('no_of_questions', ['no_of_questions'], True, 'existing source denominator hint'),
    ('bank count evidence', ['question_bank_count', 'max_marks_db', 'total_questions'], True, 'a test-level bank/total-question field should be available'),
    ('class_id', ['class_id'], False, 'optional but useful for class-level grouping'),
    ('subscriber_id', ['subscriber_id'], False, 'optional but useful for cohort grouping'),
    ('institute', ['institute'], False, 'institute context for rollups'),
    ('city', ['city'], False, 'geo context for rollups'),
    ('country', ['country'], False, 'geo context for rollups'),
]


def ncol(df: pd.DataFrame, col: str) -> pd.Series:
    if col in df.columns:
        return pd.to_numeric(df[col], errors='coerce')
    return pd.Series(np.nan, index=df.index)


def sstr(df: pd.DataFrame, col: str) -> pd.Series:
    if col in df.columns:
        return df[col].astype('string')
    return pd.Series(pd.NA, index=df.index, dtype='string')


def add_row(rows: list[dict], section: str, metric: str, value, note: str = '') -> None:
    rows.append({'section': section, 'metric': metric, 'value': value, 'note': note})


schema_rows = []
for col in EXPECTED_COLUMNS:
    present = col in raw_attempts.columns
    schema_rows.append(
        {
            'column': col,
            'present': present,
            'required_contract': any(col in item[1] and item[2] for item in LOGICAL_CONTRACT),
            'role': 'alias' if col in ALIAS_OF else ('source' if col in raw_attempts.columns and col not in ALIAS_OF else 'expected_optional'),
            'alias_of': ALIAS_OF.get(col, ''),
            'dtype': str(raw_attempts[col].dtype) if present else '',
            'null_count': int(raw_attempts[col].isna().sum()) if present else None,
            'null_rate': float(raw_attempts[col].isna().mean()) if present else None,
            'sample_non_null': raw_attempts[col].dropna().astype(str).head(3).tolist() if present else [],
        }
    )

schema_report = pd.DataFrame(schema_rows)

validation_rows: list[dict] = []

add_row(validation_rows, 'source', 'source_label', source_label, 'local raw source by default; db mode available through env vars')
add_row(validation_rows, 'source', 'source_mode', SOURCE_MODE, 'db mode uses the repo-side broad extractor')
add_row(validation_rows, 'source', 'raw_rows', int(len(raw_attempts)), 'one row per attempt')
add_row(validation_rows, 'source', 'raw_users', int(raw_attempts['user_id'].nunique()) if 'user_id' in raw_attempts.columns else 0, '')
add_row(validation_rows, 'source', 'raw_tests', int(raw_attempts['test_id'].nunique()) if 'test_id' in raw_attempts.columns else 0, '')
add_row(validation_rows, 'source', 'raw_user_test_groups', int(raw_attempts[['user_id', 'test_id']].drop_duplicates().shape[0]) if {'user_id', 'test_id'}.issubset(raw_attempts.columns) else 0, '')
add_row(validation_rows, 'source', 'created_at_min', str(pd.to_datetime(raw_attempts['created_at'], errors='coerce').min()) if 'created_at' in raw_attempts.columns else '', '')
add_row(validation_rows, 'source', 'created_at_max', str(pd.to_datetime(raw_attempts['created_at'], errors='coerce').max()) if 'created_at' in raw_attempts.columns else '', '')

for logical_name, any_of, required, note in LOGICAL_CONTRACT:
    present = [c for c in any_of if c in raw_attempts.columns]
    status = 'PASS' if present else ('FAIL' if required else 'WARNING')
    add_row(validation_rows, 'schema_contract', logical_name, status, f"present={present if present else '[]'}; {note}")

if comparison_report is not None:
    comparison = comparison_report(raw_attempts, baseline_path=BASELINE_PATH, legacy_path=LEGACY_PATH)
else:
    comparison = {'error': 'comparison_report import unavailable'}

for key, value in comparison.items():
    if isinstance(value, dict):
        for subkey, subvalue in value.items():
            add_row(validation_rows, 'reconciliation', f'{key}_{subkey}', subvalue, '')
    else:
        add_row(validation_rows, 'reconciliation', key, value, '')

finished_present = int(raw_attempts['finished_at'].notna().sum()) if 'finished_at' in raw_attempts.columns else 0
finished_missing = int(raw_attempts['finished_at'].isna().sum()) if 'finished_at' in raw_attempts.columns else int(len(raw_attempts))
activity_evidence = (
    ncol(raw_attempts, 'marks').fillna(0).gt(0)
    | ncol(raw_attempts, 'attempted_questions').fillna(0).gt(0)
    | ncol(raw_attempts, 'correct_answers').fillna(0).gt(0)
    | ncol(raw_attempts, 'time_taken').fillna(0).gt(0)
)
denominator_evidence = (
    ncol(raw_attempts, 'no_of_questions').fillna(0).gt(0)
    | ncol(raw_attempts, 'duration').fillna(0).gt(0)
    | ncol(raw_attempts, 'question_limit').fillna(0).gt(0)
    | ncol(raw_attempts, 'time_limit').fillna(0).gt(0)
    | ncol(raw_attempts, 'total_questions').fillna(0).gt(0)
    | ncol(raw_attempts, 'question_bank_count').fillna(0).gt(0)
    | ncol(raw_attempts, 'max_marks_db').fillna(0).gt(0)
)
missing_finished_usable = int((raw_attempts['finished_at'].isna() & activity_evidence & denominator_evidence).sum()) if 'finished_at' in raw_attempts.columns else 0
missing_finished_users = int(raw_attempts.loc[raw_attempts['finished_at'].isna(), 'user_id'].nunique()) if 'finished_at' in raw_attempts.columns and 'user_id' in raw_attempts.columns else 0

add_row(validation_rows, 'completion', 'finished_at_present_rows', finished_present, '')
add_row(validation_rows, 'completion', 'finished_at_missing_rows', finished_missing, '')
add_row(validation_rows, 'completion', 'finished_at_missing_but_usable_rows', missing_finished_usable, 'later candidate for unknown_but_usable in DQ')
add_row(validation_rows, 'completion', 'users_affected_by_missing_finished_at', missing_finished_users, '')
add_row(validation_rows, 'completion', 'rows_with_activity_evidence', int(activity_evidence.sum()), '')
add_row(validation_rows, 'completion', 'rows_with_denominator_evidence', int(denominator_evidence.sum()), '')

candidate_cols = ['no_of_questions', 'duration', 'question_limit', 'time_limit', 'total_questions', 'question_bank_count', 'max_marks_db']
for col in candidate_cols:
    if col in raw_attempts.columns:
        vals = ncol(raw_attempts, col)
        add_row(validation_rows, 'denominator', f'{col}_present_rows', int(vals.notna().sum()), '')
        add_row(validation_rows, 'denominator', f'{col}_gt_1000_rows', int((vals > 1000).sum()), '')
        add_row(validation_rows, 'denominator', f'marks_gt_{col}_rows', int((ncol(raw_attempts, 'marks') > vals).fillna(False).sum()), '')
    else:
        add_row(validation_rows, 'denominator', f'{col}_present_rows', 0, 'not present in current extract')

add_row(validation_rows, 'denominator', 'rows_with_any_denominator_candidate', int(denominator_evidence.sum()), '')
add_row(validation_rows, 'denominator', 'rows_with_marks_gt_any_present_candidate', int(
    (
        (ncol(raw_attempts, 'marks') > ncol(raw_attempts, 'no_of_questions'))
        | (ncol(raw_attempts, 'marks') > ncol(raw_attempts, 'duration'))
        | (ncol(raw_attempts, 'marks') > ncol(raw_attempts, 'question_limit'))
        | (ncol(raw_attempts, 'marks') > ncol(raw_attempts, 'time_limit'))
        | (ncol(raw_attempts, 'marks') > ncol(raw_attempts, 'total_questions'))
        | (ncol(raw_attempts, 'marks') > ncol(raw_attempts, 'question_bank_count'))
        | (ncol(raw_attempts, 'marks') > ncol(raw_attempts, 'max_marks_db'))
    ).fillna(False).sum()
), 'audit only; not final denominator policy')

if {'user_id', 'test_id'}.issubset(raw_attempts.columns):
    group_sizes = raw_attempts.groupby(['user_id', 'test_id']).size()
    add_row(validation_rows, 'sequence', 'one_attempt_user_test_groups', int((group_sizes == 1).sum()), '')
    add_row(validation_rows, 'sequence', 'two_plus_attempt_user_test_groups', int((group_sizes >= 2).sum()), '')
    add_row(validation_rows, 'sequence', 'max_attempts_per_user_test_group', int(group_sizes.max()), '')
    add_row(validation_rows, 'sequence', 'groups_with_duplicate_created_at', int(raw_attempts.duplicated(['user_id', 'test_id', 'created_at']).sum()) if 'created_at' in raw_attempts.columns else 0, '')
    add_row(validation_rows, 'sequence', 'created_at_present_rows', int(raw_attempts['created_at'].notna().sum()) if 'created_at' in raw_attempts.columns else 0, '')
    add_row(validation_rows, 'sequence', 'attempt_id_present_rows', int(raw_attempts['attempt_id'].notna().sum()) if 'attempt_id' in raw_attempts.columns else int(raw_attempts['test_taker_id'].notna().sum()) if 'test_taker_id' in raw_attempts.columns else 0, '')
else:
    add_row(validation_rows, 'sequence', 'one_attempt_user_test_groups', 0, 'missing user_id or test_id')
    add_row(validation_rows, 'sequence', 'two_plus_attempt_user_test_groups', 0, 'missing user_id or test_id')
    add_row(validation_rows, 'sequence', 'max_attempts_per_user_test_group', 0, 'missing user_id or test_id')

if 'class_id' in raw_attempts.columns:
    add_row(validation_rows, 'class_institute', 'class_id_present_rows', int(raw_attempts['class_id'].notna().sum()), '')
    add_row(validation_rows, 'class_institute', 'class_id_null_rate', float(raw_attempts['class_id'].isna().mean()), '')
    add_row(validation_rows, 'class_institute', 'unique_class_id_count', int(raw_attempts['class_id'].nunique(dropna=True)), '')
else:
    add_row(validation_rows, 'class_institute', 'class_id_present_rows', 0, 'not present in local fallback extract')
    add_row(validation_rows, 'class_institute', 'class_id_null_rate', 1.0, 'not present in local fallback extract')
    add_row(validation_rows, 'class_institute', 'unique_class_id_count', 0, 'not present in local fallback extract')

for col in ['subscriber_id', 'institute', 'institute_standardized', 'city', 'country']:
    if col in raw_attempts.columns:
        add_row(validation_rows, 'class_institute', f'{col}_present_rows', int(raw_attempts[col].notna().sum()), '')
        add_row(validation_rows, 'class_institute', f'{col}_null_rate', float(raw_attempts[col].isna().mean()), '')
        add_row(validation_rows, 'class_institute', f'unique_{col}_count', int(raw_attempts[col].nunique(dropna=True)), '')
    else:
        add_row(validation_rows, 'class_institute', f'{col}_present_rows', 0, 'not present in current extract')
        add_row(validation_rows, 'class_institute', f'{col}_null_rate', 1.0, 'not present in current extract')
        add_row(validation_rows, 'class_institute', f'unique_{col}_count', 0, 'not present in current extract')

caveats = []
if 'class_id' not in raw_attempts.columns or raw_attempts['class_id'].notna().sum() == 0:
    caveats.append('class_id is absent in the local fallback extract; class-level grouping will require DB mode or a later enrichment step.')
if 'subscriber_id' not in raw_attempts.columns or raw_attempts['subscriber_id'].notna().sum() == 0:
    caveats.append('subscriber_id is absent in the local fallback extract; cohort linking remains limited until DB mode is used.')
if 'delivered_result_questions' not in raw_attempts.columns:
    caveats.append('delivered_result_questions is not present in the local fallback extract; denominator diagnostics are therefore audit-only here.')
if 'answer_grade_sum_diagnostic' not in raw_attempts.columns:
    caveats.append('answer_grade_sum_diagnostic is not present in the local fallback extract; it must be derived later from response-level evidence if needed.')
if 'subject_id' not in raw_attempts.columns and 'topic_id' not in raw_attempts.columns and 'year_group' not in raw_attempts.columns:
    caveats.append('topic_id, subject_id, and year_group are intentionally not inferred in this notebook.')

baseline_rows = int(pd.read_csv(BASELINE_PATH).shape[0]) if BASELINE_PATH.exists() else None
baseline_users = int(pd.read_csv(BASELINE_PATH)['user_id'].nunique()) if BASELINE_PATH.exists() and 'user_id' in pd.read_csv(BASELINE_PATH).columns else None
legacy_rows = int(pd.read_csv(LEGACY_PATH).shape[0]) if LEGACY_PATH.exists() else None
legacy_users = int(pd.read_csv(LEGACY_PATH)['user_id'].nunique()) if LEGACY_PATH.exists() and 'user_id' in pd.read_csv(LEGACY_PATH).columns else None

raw_rows = int(len(raw_attempts))
raw_users = int(raw_attempts['user_id'].nunique()) if 'user_id' in raw_attempts.columns else 0
raw_groups = int(raw_attempts[['user_id', 'test_id']].drop_duplicates().shape[0]) if {'user_id', 'test_id'}.issubset(raw_attempts.columns) else 0

baseline_match = baseline_rows == raw_rows and baseline_users == raw_users if baseline_rows is not None and baseline_users is not None else False
legacy_match = legacy_rows == raw_rows and legacy_users == raw_users if legacy_rows is not None and legacy_users is not None else False

verdict = 'PASS'
if not baseline_match:
    verdict = 'FAIL'
elif caveats:
    verdict = 'WARNING'

add_row(validation_rows, 'verdict', 'verdict', verdict, '')
add_row(validation_rows, 'verdict', 'baseline_match_on_rows_and_users', baseline_match, '')
add_row(validation_rows, 'verdict', 'legacy_match_on_rows_and_users', legacy_match, '')
add_row(validation_rows, 'verdict', 'raw_rows', raw_rows, '')
add_row(validation_rows, 'verdict', 'raw_users', raw_users, '')
add_row(validation_rows, 'verdict', 'raw_user_test_groups', raw_groups, '')
add_row(validation_rows, 'verdict', 'suitable_as_default_dashboard_input', verdict in ['PASS', 'WARNING'], 'WARNING means usable, but with caveats')
for idx, caveat in enumerate(caveats, start=1):
    add_row(validation_rows, 'verdict', f'caveat_{idx}', caveat, '')

validation_report = pd.DataFrame(validation_rows)

schema_report.head(10), validation_report.head(20)


In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
raw_attempts.to_csv(OUTPUT_PATH, index=False)
schema_report.to_csv(SCHEMA_REPORT_PATH, index=False)
validation_report.to_csv(VALIDATION_REPORT_PATH, index=False)

print(f'Source used: {source_label}')
print(f'Raw rows: {len(raw_attempts):,}')
print(f'Unique users: {raw_attempts["user_id"].nunique():,}' if 'user_id' in raw_attempts.columns else 'Unique users: 0')
print(f'Unique tests: {raw_attempts["test_id"].nunique():,}' if 'test_id' in raw_attempts.columns else 'Unique tests: 0')
print(f'Unique user-test groups: {raw_attempts[["user_id", "test_id"]].drop_duplicates().shape[0]:,}' if {'user_id', 'test_id'}.issubset(raw_attempts.columns) else 'Unique user-test groups: 0')
print(f'Saved: {OUTPUT_PATH.relative_to(REPO_ROOT)}')
print(f'Saved: {SCHEMA_REPORT_PATH.relative_to(REPO_ROOT)}')
print(f'Saved: {VALIDATION_REPORT_PATH.relative_to(REPO_ROOT)}')
print(f'Verdict: {verdict}')
for caveat in caveats:
    print(f'- {caveat}')


## Final note

The notebook intentionally stops at the raw input contract. It does not produce published KPI, proxy-sequence, readiness, or dashboard page artifacts.
